End-to-end BPS question agent

The pipeline lives in tilik_agent.agent. This notebook calls stream_question, the same entry the API uses, and runs golden question 6. The SQL role is read-only; configure .env, PostgreSQL, and the PGVector collection before invoking the graph. No web search is used.

In [1]:
import json

from tilik_agent.agent.answer import display_chart_spec
from tilik_agent.agent.runner import stream_question
from tilik_agent.agent.schemas import State

c:\Users\USER\AppData\Local\pypoetry\Cache\virtualenvs\tilik-agent-CdBdN4pA-py3.13\Lib\site-packages\requests\__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
c:\Users\USER\AppData\Local\pypoetry\Cache\virtualenvs\tilik-agent-CdBdN4pA-py3.13\Lib\site-packages\langgraph\checkpoint\base\__init__.py:18: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer
c:\Users\USER\AppData\Local\pypoetry\Cache\virtualenvs\tilik-agent-CdBdN4pA-py3.13\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
QUESTION = (
    "Provinsi mana saja yang memiliki tiga TPT tertinggi pada Agustus 2025, "
    "dan berapa Angka Partisipasi Sekolah kelompok umur 16–18 tahun di "
    "masing-masing provinsi tersebut pada 2025?"
)

# Inner SQL agent and tool messages arrive live, tagged with their plan step.
result: State = {"question": QUESTION}
for update in stream_question(QUESTION):
    for node_name, payload in update.items():
        if node_name in ("agent", "tools"):
            for message in payload["messages"]:
                print(f"\n=== sql step {payload['step_id']}: {node_name} ===", flush=True)
                message.pretty_print()
            continue
        print(f"\n=== node: {node_name} ===", flush=True)
        if node_name == "plan_question":
            print(json.dumps({k: payload.get(k) for k in
                              ("source_route", "plan", "plan_error")},
                             ensure_ascii=False, indent=2, default=str))
        elif node_name in ("run_retrieval", "run_sql", "run_both"):
            counts: dict[str, int] = {}
            for row in payload.get("observations", []):
                key = str(row.get("dataset_id"))
                counts[key] = counts.get(key, 0) + 1
            print(json.dumps({
                "step_status": payload.get("step_status"),
                "observations_by_dataset": counts,
                "document_count": len(payload.get("documents", [])),
                "sql_error": payload.get("sql_error"),
                "sql_warnings": payload.get("sql_warnings"),
                "retrieval_error": payload.get("retrieval_error"),
                }, ensure_ascii=False, indent=2, default=str))
        elif node_name == "write_answer":
            print(payload.get("answer"))
        elif node_name == "build_chart":
            print(json.dumps(payload.get("chart"), ensure_ascii=False, indent=2, default=str))
            if payload.get("chart_error"):
                print(payload["chart_error"])
            display_chart_spec(payload.get("chart"))
        elif "answer" in payload:
            print(payload["answer"])
        result.update(payload)

print("\n=== final answer ===")
print(result["answer"])


=== node: plan_question ===
{
  "source_route": "data",
  "plan": {
    "steps": [
      {
        "id": "s1",
        "action": "sql_lookup",
        "request": "Dari tabel observasi BPS, ambil tiga provinsi dengan Tingkat Pengangguran Terbuka (TPT) tertinggi pada Agustus 2025 beserta nilai TPT-nya, lalu untuk masing-masing provinsi tersebut ambil Angka Partisipasi Sekolah (APS) kelompok umur 16-18 tahun pada 2025. Kembalikan daftar nama provinsi, nilai TPT, dan nilai APS dalam satu hasil.",
        "depends_on": []
      }
    ],
    "requested_chart": false
  },
  "plan_error": null
}

=== sql step s1: agent ===
================================== Ai Message ==================================

I'll start by finding the relevant datasets for TPT (unemployment rate) and APS (school participation rate).
Tool Calls:
  search_datasets (call_fb3f3425e577433393e97590)
 Call ID: call_fb3f3425e577433393e97590
  Args:
    query: Tingkat Pengangguran Terbuka menurut provinsi
  search_datasets 